# 301 · Trust boundaries experiment

Companion to [Trust boundaries](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/301/trust-boundaries/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/301/trust_boundaries.ipynb)

**Question:** May these bytes use a **language-native** codec, or must they be **portable**?

> **Honesty banner:** notebook experiments implement the article **Experiments** blocks. They do not replace suite Results. Timings/sizes are illustrative.



In [ ]:
from dataclasses import dataclass
from typing import List


@dataclass
class Hop:
    name: str
    untrusted_or_other_lang: bool
    multi_tenant: bool
    long_lived: bool
    notes: str = ""


def policy_for_path(hops: List[Hop]) -> str:
    for h in hops:
        if h.untrusted_or_other_lang or h.multi_tenant:
            return "PORTABLE required (native disqualified on this path)"
        if h.long_lived:
            return "PORTABLE strongly preferred (native lock-in / skew risk)"
    return "NATIVE allowed only if same-runtime, trusted peers, documented threat model"


paths = {
    "public HTTP body": [Hop("edge API", True, True, False)],
    "Redis cache same binary only": [Hop("redis", False, False, False, "private net")],
    "S3 model blob multi-team": [Hop("object store", False, True, True)],
    "queue Python → Go consumer": [Hop("kafka", True, False, True)],
}
for name, hops in paths.items():
    print(f"{name:32} → {policy_for_path(hops)}")



## Optional: size is not a trust argument



In [ ]:
import json, pickle

obj = {"user_id": 1, "roles": ["admin", "ops"], "prefs": {f"k{i}": i for i in range(20)}}
j = json.dumps(obj).encode()
p = pickle.dumps(obj)
print(f"JSON {len(j)} bytes; pickle {len(p)} bytes")
print("Faster/smaller native still loses if any hop fails the trust test.")



## Metrics (from the article)

| Signal | Role |
|--------|------|
| Trust-domain crossing | **Primary** |
| Consumer language set | Forces portable if >1 family |
| Suite speed | Secondary, only among policy-allowed codecs |

**Conclusion style:** “Queue hop is multi-tenant → portable only; native pickle rejected despite size.”

